# Unit 2: Data Transformation

**Luis Cuevas**
Purdue Global
IN498: Bachelor's Capstone in Analytics
Dr. Larson
April 7, 2025


## 1. Converting Raw Files to CSV

The nine uncleaned raw data files, each contained within a CSV format, included fourteen separate fields for each record. However, instead of being in individual columns, these fourteen field values for each record existed as an individual concatenated string in one column. There also was no header row included on this structure. Since there was no header row, this structure violated the tidy data principle defined by Wickham (2014). In addition to violating the tidy data principle, this structure had other issues. Four files (September, October, November and December 2019) were created in UTF-16 Little Endian format with CRLF line endings, whereas the remainder of the five files were created in standard UTF-8 format.

All of the files were processed through python and the pandas library. The pandas read_csv() method was utilized to process each file with None for the "header" argument and a pre-determined list of 14 column names that corresponded to the correct position of the 14 column values. All of the raw data from the single column was distributed into individual cells using str.split(",") with expand=True. The structured DataFrames produced after processing each file were then preserved without modifying any of the original values using the to_csv() method and the index=False parameter.


## 2. Process Summary: Raw File Conversion

Firstly, there were no headers in these unprocessed raw files so they were read in as one string that contained all the values in each row in a comma-separated format. Secondly, we determined the type of encoding using python's file utility. Four of them were encoded in UTF-16 little-endian and five of them were encoded in UTF-8. Thirdly, we read every file into a pandas DataFrame using pandas' read_csv function with a header parameter set to None. We also passed it an array of 14 pre-defined column names that represent the columns Date, App, Country, Installers and the retained and retention rates for day windows of 1, 7, 15, 30, and 60 days. Lastly, after splitting each row using the str.split() method we wrote those DataFrames to a new CSV file named using pandas.to_csv function with the index parameter set to False. No cleaning was done on our data at this point because like Wickham & Grolemund (2017) recommend, it is better to keep raw data ingestion separate from the cleaning of your data.


## 3. Python Explorer: Individual Files

The Python explorer script *IN498_Unit2_Student_Explorer.py* was developed to process each of the nine individual CSV files. For each file, the script prints the first 10 rows using *df.head(10)*, the shape using *df.shape*, the statistical description using *df.describe()*, the data counts using *df.count()*, the mean and mode for retained installers at 1, 7, 15, and 30 days. All output is written to *IN498_Explorer.txt*.

In [ ]:
import pandas as pd
import sys

files = [
    "retained_installers_com.foo.bar_201904_country.csv",
    "retained_installers_com.foo.bar_201905_country.csv",
    "retained_installers_com.foo.bar_201906_country.csv",
    "retained_installers_com.foo.bar_201907_country.csv",
    "retained_installers_com.foo.bar_201908_country.csv",
    "retained_installers_com.foo.bar_201909_country.csv",
    "retained_installers_com.foo.bar_201910_country.csv",
    "retained_installers_com.foo.bar_201911_country.csv",
    "retained_installers_com.foo.bar_201912_country.csv",
]

cols = ['Date','App','Country','Installers','Retained_1d','Retention_rate_1d',
        'Retained_7d','Retention_rate_7d','Retained_15d','Retention_rate_15d',
        'Retained_30d','Retention_rate_30d','Retained_60d','Retention_rate_60d']

output_lines = []

def log(text=""):
    """Print to console and collect for file output."""
    print(text)
    output_lines.append(str(text))

# process each file
for filename in files:
    log("=" * 70)
    log(f"FILE: {filename}")
    log("=" * 70)

    try:
        # read raw single-column data, try utf-8 first then utf-16
        try:
            raw = pd.read_csv(filename, header=None, names=['raw'], encoding='utf-8')
        except UnicodeDecodeError:
            raw = pd.read_csv(filename, header=None, names=['raw'], encoding='utf-16')

        # split each row on comma into proper columns
        df = raw['raw'].str.split(',', expand=True)
        df.columns = cols

    except FileNotFoundError:
        log(f"  ERROR: {filename} not found. Skipping.\n")
        continue

    # enforce numeric types
    numeric_cols = ["Installers", "Retained_1d", "Retained_7d",
                    "Retained_15d", "Retained_30d"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    # first 10 rows
    log("\n--- First 10 Rows ---")
    log(df.head(10).to_string())

    # shape
    log("\n--- Shape (rows, columns) ---")
    log(str(df.shape))

    # description
    log("\n--- Description ---")
    log(df.describe().to_string())

    # counts
    log("\n--- Counts ---")
    log(df.count().to_string())

    # mean – retained 1, 7, 15, 30 days
    log("\n--- Mean: Installers Retained 1, 7, 15, 30 Day(s) ---")
    log(df[["Retained_1d", "Retained_7d", "Retained_15d", "Retained_30d"]].mean().to_string())

    # mode – retained 1, 7, 15, 30 days
    log("\n--- Mode: Installers Retained 1, 7, 15, 30 Day(s) ---")
    log(df[["Retained_1d", "Retained_7d", "Retained_15d", "Retained_30d"]].mode().iloc[0].to_string())

    log("")

# write results to text file
output_path = "IN498_Explorer.txt"
with open(output_path, "w") as f:
    f.write("\n".join(output_lines))

print(f"\nResults written to {output_path}")

FILE: retained_installers_com.foo.bar_201904_country.csv

--- First 10 Rows ---
         Date          App Country  Installers  Retained_1d Retention_rate_1d  Retained_7d Retention_rate_7d  Retained_15d Retention_rate_15d  Retained_30d Retention_rate_30d Retained_60d Retention_rate_60d
0  2019/04/01  com.foo.bar      AR           1            1             1.000            1             1.000             1              1.000             1              1.000            0              0.000
1  2019/04/01  com.foo.bar      AU           3            2             0.667            2             1.000             2              1.000             1              0.500            1              0.500
2  2019/04/01  com.foo.bar      AZ           1            0             0.000            0             0.000             0              0.000             0              0.000            0              0.000
3  2019/04/01  com.foo.bar      BR           1            0             0.000            0  

## 4. Individual File Analysis Results

### Shape of Each Data Frame

The following rows and columns were identified in the data output for all of the monthly datasets as follows:

- April 2019 (889 Rows; 14 Columns)
- May 2019 (895 Rows; 14 Columns)
- June 2019 (790 Rows; 14 Columns)
- July 2019 (727 Rows; 14 Columns)
- August 2019 (778 Rows; 14 Columns)
- September 2019 (560 Rows; 14 Columns)
- October 2019 (582 Rows; 14 Columns)
- November 2019 (595 Rows; 14 Columns)
- December 2019 (601 Rows; 14 Columns)

Each file had exactly fourteen columns to match the specified schema.

### Missing Data by Dataset

The count for each file shows that there are the same number of records overall as when you add up the record counts for each column across all of the 14 columns. Therefore, it appears that there are no missing values identified for each of the monthly data files. Nevertheless, the November 2019 file includes at least one row with a blank entry instead of a country code for the Country field. As shown by the first ten entries of the output, this was clearly evident; however, the count() method did not identify the blank as missing since it does not recognize an empty string as NA in pandas (McKinney, 2022), which would require rectification in the curation process.

### Mean and Mode for Retention Windows

The following displays the mean retained installer counts by monthly file:

- April 2019 — (1 day) = 2.49 | (7 days) = 1.46 | (15 days) = 1.14 | (30 days) = .97
- May 2019 — (1 day) = 2.45 | (7 days) = 1.53 | (15 days) = 1.21 | (30 days) = 1.03
- June 2019 — (1 day) = 2.42 | (7 days) = 1.55 | (15 days) = 1.24 | (30 days) = 1.07
- July 2019 — (1 day) = 2.57 | (7 days) = 1.45 | (15 days) = 1.10 | (30 days) = .95
- August 2019 — (1 day) = 2.78 | (7 days) = 1.55 | (15 days) = 1.20 | (30 days) = 1.04
- September 2019 — (1 day) = 2.03 | (7 days) = 1.36 | (15 days) = 1.11 | (30 days) = .98
- October 2019 — (1 day) = 3.08 | (7 days) = 1.81 | (15 days) = 1.39 | (30 days) = 1.20
- November 2019 — (1 day) = 2.43 | (7 days) = 1.57 | (15 days) = 1.24 | (30 days) = 1.11
- December 2019 — (1 day) = 2.38 | (7 days) = 1.43 | (15 days) = 1.12 | (30 days) = .96

The mode for all four retention windows (1, 7, 15, and 30 days) is 0 across all nine files.

### Conclusions: Initial Analysis

A commonality in each of the nine files across months is that the mode for every retention window was zero. In other words, the most frequent occurrence of retainers at any given time was zero. This supports previous research in Mobile Application Analytics that has shown that most new installs are limited to single session activity, resulting in minimal conversion to retained users (Handel, 2013). Each of the mean values demonstrates a distinct retention funnel pattern for each month, from highest 1 day retention, declining as we move down the funnel into 7 days, 15 days, and 30 days. For example, October 2019 had the highest 1 day mean at 3.08, while September 2019 had the lowest at 2.03, suggesting variability in the monthly means based on install volume and possibly geographic location. Additionally, September 2019 had fewer total rows than any of the other months (560), which further suggests it may have been a lower volume month. The two major Data Quality Issues that need to be addressed prior to conducting final analysis include an "empty string" in the Country column for November 2019, along with non-consistent formatting in the date columns throughout each file.


## 5. Merging Nine CSV Files into a Final Dataset

All nine separate csv files were combined into an existing DataFrame called IN498_Final.csv, utilizing the pandas pd.concat() function with ignore_index=True that simply stacks all of the nine dataframes on top of each other without changing any of the information in them. There is no cleanup or curation done at this time; again, we are merging the data files "as-is" as per the directions provided. The resultant merged file will have 6417 rows and 14 columns.

## 6. Python Explorer: Final Dataset

The last Explorer Script (IN498_Unit2_Student_Final_Explorer.py) uses the same structure to analyze the combined data set from the IN498_Final.csv file that the previous Individual Explorer Scripts did; i.e., first ten lines of the data, the total number of observations (shape), a description of the data set, counts for each retention window, means for all variables by retention window, and modes for all variables by retention window. All output from this script will be sent to the IN498_Final_Explorer.txt file.

In [ ]:
import pandas as pd

# original filenames as uploaded
files = [
    "retained_installers_com.foo.bar_201904_country.csv",
    "retained_installers_com.foo.bar_201905_country.csv",
    "retained_installers_com.foo.bar_201906_country.csv",
    "retained_installers_com.foo.bar_201907_country.csv",
    "retained_installers_com.foo.bar_201908_country.csv",
    "retained_installers_com.foo.bar_201909_country.csv",
    "retained_installers_com.foo.bar_201910_country.csv",
    "retained_installers_com.foo.bar_201911_country.csv",
    "retained_installers_com.foo.bar_201912_country.csv",
]

cols = ['Date','App','Country','Installers','Retained_1d','Retention_rate_1d',
        'Retained_7d','Retention_rate_7d','Retained_15d','Retention_rate_15d',
        'Retained_30d','Retention_rate_30d','Retained_60d','Retention_rate_60d']

# parse and merge all files
dfs = []
for filename in files:
    try:
        raw = pd.read_csv(filename, header=None, names=['raw'], encoding='utf-8')
    except UnicodeDecodeError:
        raw = pd.read_csv(filename, header=None, names=['raw'], encoding='utf-16')
    df = raw['raw'].str.split(',', expand=True)
    df.columns = cols
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv("IN498_Final.csv", index=False)
print(f"Merged file saved: IN498_Final.csv  |  Shape: {final_df.shape}")

# final explorer
numeric_cols = ["Installers", "Retained_1d", "Retained_7d",
                "Retained_15d", "Retained_30d"]
final_df[numeric_cols] = final_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

output_lines = []

def log(text=""):
    print(text)
    output_lines.append(str(text))

log("=" * 70)
log("FINAL DATASET EXPLORER  |  File: IN498_Final.csv")
log("=" * 70)

log("\n--- First 10 Rows ---")
log(final_df.head(10).to_string())

log("\n--- Shape (rows, columns) ---")
log(str(final_df.shape))

log("\n--- Description ---")
log(final_df.describe().to_string())

log("\n--- Counts ---")
log(final_df.count().to_string())

log("\n--- Mean: Installers Retained 1, 7, 15, 30 Day(s) ---")
log(final_df[["Retained_1d", "Retained_7d", "Retained_15d", "Retained_30d"]].mean().to_string())

log("\n--- Mode: Installers Retained 1, 7, 15, 30 Day(s) ---")
log(final_df[["Retained_1d", "Retained_7d", "Retained_15d", "Retained_30d"]].mode().iloc[0].to_string())

with open("IN498_Final_Explorer.txt", "w") as f:
    f.write("\n".join(output_lines))

print("\nResults written to IN498_Final_Explorer.txt")

Merged file saved: IN498_Final.csv  |  Shape: (6417, 14)
FINAL DATASET EXPLORER  |  File: IN498_Final.csv

--- First 10 Rows ---
         Date          App Country  Installers  Retained_1d Retention_rate_1d  Retained_7d Retention_rate_7d  Retained_15d Retention_rate_15d  Retained_30d Retention_rate_30d Retained_60d Retention_rate_60d
0  2019/04/01  com.foo.bar      AR           1            1             1.000            1             1.000             1              1.000             1              1.000            0              0.000
1  2019/04/01  com.foo.bar      AU           3            2             0.667            2             1.000             2              1.000             1              0.500            1              0.500
2  2019/04/01  com.foo.bar      AZ           1            0             0.000            0             0.000             0              0.000             0              0.000            0              0.000
3  2019/04/01  com.foo.bar      BR         

## 7. Final Dataset Analysis Results

### Shape of the Final Data Frame

The final merged dataset has a shape of 6,417 rows and 14 columns, representing the complete combined records from all nine monthly files.

### Data Counts for the Final Dataset

All 14 columns in the final merged dataset return a complete count of 6,417, indicating that no null values were detected across any column in the merged dataset. However, the Country field, while fully counted at 6,417, contains at least one blank string value inherited from the November 2019 file. As previously noted, pandas does not recognize an empty string as NaN, so this did not reduce the count but still represents a data quality issue requiring remediation (McKinney, 2022). Additionally, although the Date column is fully populated, it contains two inconsistent date formats that will need to be standardized during the curation process.

### Mean and Mode for Final Dataset Retention Windows

The mean retained installer counts for the final merged dataset are:

- 1-day retention 2.518
- 7-day retention 1.522
- 15-day retention 1.192
- 30-day retention 1.030

The mode for all four retention windows is 0, consistent with the individual file results.


## 8. Data Curation, Cleaning, and Transformation

### Date Format Standardization

In total, five of the files utilized the YYYY/MM/DD format (April, June, August, October, and December 2019), while the remaining four used the YYYY-MM-DD format (May, July, September, and November 2019). The Date column was standardized to YYYY-MM-DD using both the pd.to_datetime() function and strftime('%Y-%m-%d'). This is an acceptable approach for resolving mixed date formatting in table-based datasets (McKinney, 2022).

### Missing Country Values

In the November 2019 dataset, there is one entry with an empty Country field. This field will be replaced with "Unknown" rather than removed, in order to preserve its associated numeric fields. The use of the string "Unknown" helps maintain data integrity by retaining all numeric values while labeling the record as geographically unclassifiable, consistent with the general principle of minimizing information loss when curating datasets (Wickham & Grolemund, 2017).

### Missing 60-Day Retention Values

The final explorer confirmed that all 6,417 rows contain complete values for both Retained_60d and Retention_rate_60d, meaning no missing values were detected in the 60-day retention columns. As a result, no exclusion or imputation was necessary for this window, and all rows remain available for use across all retention analyses (Van den Broeck et al., 2005).

### Data Type Enforcement

All numeric columns were converted using *pd.to_numeric(errors='coerce')* to enforce consistent data types across the merged dataset, ensuring non-numeric strings are converted to NaN rather than causing runtime errors during statistical operations (McKinney, 2022).


## 9. Handling Missing Data: Process Explanation

The method used to address the missing data in the current dataset involved identifying and addressing each specific data quality problem individually rather than deleting entire rows based on an initial assessment of data completeness. This approach allowed for the preservation of as much valid data as possible. Van den Broeck et al. (2005) recommend evaluating missingness in context before deciding upon a missing data treatment plan, distinguishing among three types of missingness: missing completely at random, missing at random, and missing not at random.

The decision to use "Unknown" for imputation rather than delete the row containing the missing Country value for November 2019 was due to the presence of complete installer and retention counts. Deleting this single row would have created an artificial gap in the November 2019 time series that could result in distortion when computing monthly aggregate totals. Therefore, using a categorical placeholder to replace the missing country appears reasonable given that the missingness does not appear to be systematic in nature (Salgado et al., 2016).

Regarding the 60-day retention columns, the final explorer confirmed that Retained_60d and Retention_rate_60d are fully populated across all 6,417 rows, so no exclusion or imputation was required for this window. This outcome reflects the overall completeness of the raw data across all retention intervals.

Regarding the inconsistencies in date format, a conversion routine using pd.to_datetime() successfully standardized all date entries into a consistent format without requiring the removal of any individual rows. The overarching goal throughout this curatorial process was to retain as many records as possible by implementing targeted imputations and scoped exclusions where needed, given the aggregated nature of this dataset and the relatively few records necessary to compute trends across countries per month (Wickham & Grolemund, 2017).


## References

Handel, M. J. (2013). *The growth of mobile app markets*. SSRN. https://doi.org/10.2139/ssrn.2242147

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media.

Salgado, C. M., Azevedo, C., Proenca, H., & Vieira, S. M. (2016). Missing data. In R. Hoerbst, A. Balogh, P., & Sandner, F. (Eds.), *Secondary analysis of electronic health records* (pp. 143–162). Springer. https://doi.org/10.1007/978-3-319-43742-2_13

Van den Broeck, J., Cunningham, S. A., Eeckels, R., & Herbst, K. (2005). Data cleaning: Detecting, diagnosing, and editing data abnormalities. *PLOS Medicine, 2*(10), e267. https://doi.org/10.1371/journal.pmed.0020267

Wickham, H. (2014). Tidy data. *Journal of Statistical Software, 59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10

Wickham, H., & Grolemund, G. (2017). *R for data science: Import, tidy, transform, visualize, and model data*. O'Reilly Media. https://r4ds.had.co.nz